# Лекција 10 - AI агенти у производњи

У овој лекцији ћете научити **шаблоне за производњу** за AI агенте користећи Microsoft Agent Framework (Python). Обухватамо:

- **Observability** — додавање праћења времена и логовања интеракција агента
- **Evaluation** — коришћење агента-оцењивача за оцењивање квалитета одговора
- **Cost management** — стратегије за оптимизацију токена и избор модела

Сценарио је **туристички агент** који помаже корисницима да планирају путовања, са надгледањем и евалуацијом као додатним слојевима.


## Подешавање


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Разматрања за производњу

Премештање AI агената из прототипова у производњу захтева пажљиву пажњу на три стуба:

1. **Видљивост** — Потребна вам је видљивост у то шта агент ради, колико му то траје и које алате позива. Без праћења и логовања, решавање грешака у продукцији је готово немогуће.

2. **Евалуација** — Аутоматизоване контроле квалитета обезбеђују да одговори агента остану тачни, потпуни и корисни током времена. Агент за евалуацију може оцењивати одговоре према дефинисаним критеријумима.

3. **Управљање трошковима** — Употреба токена директно утиче на трошкове. Стратегије попут оптимизације промпта, избора модела и кеширања помажу да се расходи држе под контролом без жртвовања квалитета.


## Изградња агента за надгледање

Дефинишемо алате за путовање и омотавамо позив агента мерењем времена како бисмо могли да пратимо латенцију. У продукцији бисте интегрисали OpenTelemetry или сличан бекенд за праћење (tracing).


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Обрасци процене

Уобичајен производни образац је коришћење другог агента као **оценитељ**. Оценитељ оцењује одговор примарног агента према унапред дефинисаним критеријумима као што су потпуност, тачност и корисност.

Ово омогућава:
- Аутоматизоване контроле квалитета пре него што одговори стигну до корисника
- Откривање регресије када се промпти или модели промене
- Континуирано праћење перформанси агента током времена


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегије управљања трошковима

Контрола трошкова је кључна за продукционе AI агенте. Ево кључних стратегија:

| Стратегија | Опис |
|---|---|
| **Оптимизација prompta** | Држите системске инструкције сажетим. Уклоните редундантни контекст како бисте смањили улазне токене. |
| **Избор модела** | Користите мање, јефтиније моделе (нпр. GPT-4o-mini) за једноставне задатке као што су класификација или екстракција, а веће моделе резервишите за сложено резоновање. |
| **Kešирање** | Kešирајте резултате алатки и честе упите да бисте избегли редундантне API позиве. |
| **Budžeti tokena** | Подесите ограничења `max_tokens` како бисте спречили неочекивано дуге одговоре. |
| **Груписање** | Групишите више корисничких упита у један API позив где је то могуће. |

У пракси, слојевити приступ добро функционише: усмерите једноставне захтеве ка брзом, јефтином моделу и ескалирајте само сложене упите ка способнијем.


## Резиме

У овој лекцији сте научили како да:

1. **Додајте видљивост** у интеракције агената коришћењем временских ознака и логовања, постављајући темељ за праћење и надзор.
2. **Процените одговоре агента** аутоматски помоћу агента-оцењивача који оцењује потпуност, тачност и корисност.
3. **Управљајте трошковима** кроз оптимизацију упита, избор модела, кеширање и буџете токена.

Ови производни обрасци помажу да ваши агенти вештачке интелигенције буду поуздани, мерљиви и економични у великом обиму.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен помоћу услуге за превођење која користи вештачку интелигенцију [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да обезбедимо тачност, молимо имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било какве неспоразуме или погрешна тумачења која проистекну из употребе овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
